# 69) Repeated Measures ANOVA (Tekrarlı Ölçümlerde ANOVA)
Bağımlı Örneklem T Testi'nde (Konu 61), **aynı bireylerin 2 farklı zamanda** ölçüldüğü durumu görmüştük (öncesi-sonrası). Repeated Measures ANOVA, bunun **genişletilmiş hali** — aynı bireylerin **3 veya daha fazla kez** (2'den fazla zaman noktasında/koşulda) ölçüldüğü durumları test eder.

## Ne Zaman Kullanılır?
Aynı kişiler/birimler, **birden fazla kez** ölçüldüğünde. Örnek: "Aynı 100 müşterinin, 4 farklı ay boyunca (Ocak, Şubat, Mart, Nisan) memnuniyet puanı nasıl değişti?" — burada 4 zaman noktası var, hepsi **aynı müşterilerden** geliyor.

## Neden Normal (Bağımsız) ANOVA Değil?
Eğer bu 4 ayı, birbirinden bağımsız 4 farklı grup gibi ele alıp normal One-Way ANOVA kullansaydık, **aynı kişilerin tekrar tekrar ölçüldüğü** gerçeğini göz ardı etmiş olurduk — bu, Konu 61'de bağımsız/bağımlı T testi ayrımında gördüğümüz sorunun ANOVA'daki karşılığı. Aynı kişinin farklı zamanlardaki ölçümleri **birbiriyle ilişkilidir** (bağımsız değildir), bu ilişkiyi hesaba katmazsak, hem yanlış sonuçlar hem de kaybedilen istatistiksel güç (Power) ile karşılaşırız.

## Ek Bir Varsayım: Sphericity (Küresellik)
Repeated Measures ANOVA'nın, normal ANOVA'nın 3 varsayımına ek olarak, **Sphericity (Küresellik)** adında özel bir varsayımı daha vardır — ölçüm çiftleri arasındaki farkların varyansının, tüm çiftlerde **eşit** olması gerektiğini söyler. Bu varsayım ihlal edilirse, **Greenhouse-Geisser düzeltmesi** gibi yöntemlerle sonuç düzeltilir. (Bu, oldukça ileri seviye bir detay — şimdilik sadece böyle bir varsayımın var olduğunu bilmemiz yeterli, derinlemesine girmiyoruz.)

**Sphericity testi (Mauchly's W) sonucu:**
1. p_spher > 0.05 → Sphericity sağlanıyor → p_unc kullanırız.
2. p_spher < 0.05 → Sphericity ihlal edilmiş → p_GG_corr (düzeltilmiş) kullanırız.

## Python'da Kullanımı
`scipy`'de yoktur — `statsmodels` veya `pingouin` kütüphaneleri kullanılır:
```python
import pingouin as pg
sonuc = pg.rm_anova(data=df, dv='Memnuniyet', within='Ay', subject='Musteri_ID')
```
- `dv` (dependent variable) → bağımlı değişken (ölçülen şey)
- `within` → tekrarlanan faktör (zaman/koşul)
- `subject` → her bireyi ayırt eden ID sütunu (KRİTİK — bu olmadan, hangi ölçümün hangi kişiye ait olduğu bilinmez)

In [10]:
import numpy as np
import pandas as pd
import pingouin as pg
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.8f}')
np.random.seed(42)

n_musteri = 25
musteri_id = np.arange(1, n_musteri+1)

# Aynı müşterilerin 4 ay boyunca memnuniyeti - zamanla artan bir trend kurguluyoruz

# H0: 4 ay boyunca ortalama memnuniyet puanı arasında fark yoktur.
# H1: En az bir ay, diğerlerinden farklı bir ortalama memnuniyete sahiptir.
ocak = np.random.normal(loc=65, scale=8, size=n_musteri)
subat = np.random.normal(loc=70, scale=8, size=n_musteri)
mart = np.random.normal(loc=76, scale=8, size=n_musteri)
nisan = np.random.normal(loc=82, scale=8, size=n_musteri)

df_genis = pd.DataFrame({
    'Musteri_ID': musteri_id,
    'Ocak': ocak,
    'Subat': subat,
    'Mart': mart,
    'Nisan': nisan
})

df_uzun = pd.melt(
    frame=df_genis,
    id_vars='Musteri_ID',
    value_vars=['Ocak', 'Subat', 'Mart', 'Nisan'],
    var_name='Aylar',
    value_name='Memnuniyet Skoru'
)
ocak_ort = df_genis['Ocak'].mean()
subat_ort = df_genis['Subat'].mean()
mart_ort = df_genis['Mart'].mean()
nisan_ort = df_genis['Nisan'].mean()
print(f"Ocak ayının memnuniyet puan ortalaması: {ocak_ort}")
print(f"Şubat ayının memnuniyet puan ortalaması: {subat_ort}")
print(f"Mart ayının memnuniyet puan ortalaması: {mart_ort}")
print(f"Nisan ayının memnuniyet puan ortalaması: {nisan_ort}")
sonuc = pg.rm_anova(data=df_uzun, dv='Memnuniyet Skoru', within='Aylar', subject='Musteri_ID', correction=True)
posthoc = pg.pairwise_tests(data=df_uzun, dv='Memnuniyet Skoru', within='Aylar', subject='Musteri_ID', padjust='bonferroni')
print()
print('Tekrarlı Ölçümler ANOVA ve Post-Hoc Analizi Sonuçları')
print(sonuc.head().to_string())
posthoc

Ocak ayının memnuniyet puan ortalaması: 63.69193553073527
Şubat ayının memnuniyet puan ortalaması: 67.70048198516649
Mart ayının memnuniyet puan ortalaması: 76.8487516954527
Nisan ayının memnuniyet puan ortalaması: 81.43574223203453

Tekrarlı Ölçümler ANOVA ve Post-Hoc Analizi Sonuçları
  Source  ddof1  ddof2           F      p_unc  p_GG_corr        ng2        eps  sphericity    W_spher    p_spher
0  Aylar      3     72 36.21825060 0.00000000 0.00000000 0.49453934 0.90565529        True 0.83567162 0.53842263


,Contrast,A,B,Paired,Parametric,T,dof,alternative,p_unc,p_corr,p_adjust,BF10,hedges
0,Aylar,Mart,Nisan,True,True,-2.16523247,24.00000000,two-sided,0.04053255,0.24319527,bonferroni,1.514,-0.64222103
1,Aylar,Mart,Ocak,True,True,5.98334551,24.00000000,two-sided,0.00000355,0.00002130,bonferroni,5574.243,1.66554012
2,Aylar,Mart,Subat,True,True,5.49731018,24.00000000,two-sided,0.00001187,0.00007121,bonferroni,1849.493,1.17635024
3,Aylar,Nisan,Ocak,True,True,9.00152503,24.00000000,two-sided,0.00000000,0.00000002,bonferroni,3.285e+06,2.53337887
4,Aylar,Nisan,Subat,True,True,8.24735851,24.00000000,two-sided,0.00000002,0.00000011,bonferroni,7.314e+05,2.00064035
5,Aylar,Ocak,Subat,True,True,-2.21847335,24.00000000,two-sided,0.03623463,0.21740778,bonferroni,1.656,-0.52399766


### Sonuç
Öncelikle ocak, şubat, mart ve nisan aylarına ait müşterilerin memnuniyet puanlarını bir geniş data frame yapısında girdik. Bu yapıda her bir satır bir kişinin ocak, şubat, mart ve nisan aylarındaki memnuniyet puanlarını temsil ediyordu pingouin kütüphanesi tek yönlü tekrarlı ölçümlerde ANOVA için hem geniş hem de uzun formatta dataframe yapısı kabul ediyor, biz de uzun formatta dataframe yapısını göstermek adına veriyi melt fonksiyonu ile uzun dataframe formatına dönüştürdük. Bu durumda artık her bir satır kişilerin tek tek aylık gözlemlerini temsil etti. Ardından uyguladığımız tekrarlı ölçümler ANOVA testi sonuçlarına göre, en az bir ay diğerlerinden anlamlı olarak farklı memnuniyet puanına sahiptir. Yukarıda bahsettiğimiz küresellik varsayımının sonuçlarını görmek için ise rm_anova fonksiyonuna `correction=True` parametresi ekledik. Küresellik varsayımının hipotez yapısı da normallik varsayımı gibi p > 0.05 ise veriler küresellik yapısını sağlıyor, p<0.05 ise veriler küresellik varsayımını sağlamıyordur. Veriler küresellik varsayımını sağlıyor bunu sphericity = True yapısından anlıyoruz. False olsaydı bu durum küresellik varsayımı sağlanmıyor deyip düzeltilmiş p değerini (p_GG_corr) kısmını dikkate almamız gerekirdi ancak küresellik varsayımı sağlandığı için düzeltilmemiş p değerini kullanabiliriz (p_unc). En sonda çoklu karşılaştırma testi yaptık, burada 6 tane karşılaştırma testi olduğu için en az bir testte Tip 1 Hata (fark yokken fark var demek) bulma olasılığımız (1 - 0.95^6) = ~%26.49'a çıkıyor. Bu durumda düzeltilmiş p değerini kullanmalıyız. Nedir bu düzeltilmiş p değeri ? Bu p-value, toplam hata payını (Family-Wise Error Rate) yeniden %5 seviyesine çekmek için ham p değerlerinin matematiksel bir ceza yöntemiyle büyütülmüş/düzeltilmiş halidir. Dolayısıyla p_corr kısmına baktığımızda ocak ayından mart ayına, şubat ayından mart ayına, ocak ayından nisan ayına ve şubat ayından nisan ayına anlamlı şekilde fark vardır.